In [ ]:
%matplotlib inline
import os
os.environ['PY3_PROD'] = '1'
%load_ext autoreload
%autoreload 2
os.system('kinit')

In [ ]:
import numpy as np
import pandas as pd
import re
import datetime
import matplotlib
from pycmqlib3.utility import dbaccess, dataseries, misc
from pycmqlib3.analytics.tstool import *
from pycmqlib3.analytics.btmetrics import *
from pycmqlib3.analytics.backtest_utils import *
from pycmqlib3.strategy.signal_repo import *

In [ ]:
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)
matplotlib.rcParams['figure.figsize'] = (12, 8)
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>div.output_scroll { height: 44em; }</style>"))

In [ ]:
import pickle
data_file = open("C:/dev/data/optiver_processed.pkl", 'rb')
data_dict = pickle.load(data_file)

In [ ]:
DATA_DIR = './data/'
print("\nLoading commodity information...")
commodity_info = pd.read_csv(DATA_DIR + 'commodity_info.csv')

sector_map = commodity_info.groupby("sector")["commodity"].apply(list).to_dict()

In [ ]:
fut_data = data_dict['hist_data']

In [ ]:
adf = pd.DataFrame(data_dict['roll_adj'])
asset_list = adf.columns
us_assets = [asset for asset in asset_list if asset[:2] == 'US']
cn_assets = [asset for asset in asset_list if asset[:2] == 'CN']
cn_index = adf[cn_assets].dropna(how='all').index
us_index = adf[us_assets].dropna(how='all').index

In [ ]:
DATA_DIR = './data/'
data_files = [
    'economic_weekly', 'economic_monthly', 'fx_data',
    'weather_weekly', 'supply_chain_weekly', 'geopolitical_weekly',
    'sentiment_daily', 'shipping_weekly', 'energy_weekly',
    'agricultural_weekly', 'industrial_weekly', 'financial_daily'
]

alt_data_list = []
alt_data = {}
for file_name in data_files:
    df = pd.read_csv(DATA_DIR + file_name + '.csv')
    df['date'] = pd.to_datetime(df['date'])
    df = df.set_index('date')
    alt_data_list.append(df)
    alt_data[file_name] = df
spot_df = pd.concat(alt_data_list, axis=1)

In [ ]:
sector_map = {
    'economic_weekly': asset_list, 
    'financial_daily': asset_list,
    'economic_monthly': asset_list,
    'weather_weekly':  asset_list,
    'supply_chain_weekly':  asset_list, 
    'geopolitical_weekly':  asset_list,
    'industrial_weekly': sector_map['BaseMetals'] + sector_map['Ferrous']

}

In [ ]:
start_date = adf.index[0]
end_date = adf.index[-1]
cdates = pd.date_range(start=start_date, end=end_date, freq='D')


In [ ]:
vol_win=20
pnl_tenors = ['6m', '1y', '2y', '3y', '4y', '5y', '6y', '7y', '8y']
insample = "2026-01-01"

empiric_assets = asset_list #["US_Crude", "US_HeatingOil", "US_Gasoline"]

df_pxchg = adf[empiric_assets].dropna(how='all').ffill().pct_change().fillna(0)
vol_df = pd.DataFrame(index=df_pxchg.index, columns=df_pxchg.columns)
for asset in empiric_assets:
    vol_df[asset] = df_pxchg[asset].replace(0, np.nan).dropna().rolling(vol_win).std()

In [ ]:
spot_df["us_hy_ig_spd"] = spot_df["us_hy_spread"]/spot_df["us_ig_spread"]
spot_df["us_10y_2y_spd"] = spot_df["us_10y_yield"] - spot_df["us_2y_yield"]
spot_df['china_m2_m1_spd'] = spot_df['china_m2_growth_yoy'] - spot_df['china_m1_growth_yoy']

In [ ]:
# spot_df['us_oil_inv'] = spot_df[['us_crude_inventory', 'us_gasoline_inventory', 'us_distillate_inventory']].sum(axis=1)
# spot_df["us_heatingoil_inventory"] = spot_df['us_distillate_inventory']
# spot_df['']=  spot_df['us_nat_gas_storage']

In [ ]:
cutoff='2018-01-01'
shift_holdings = 1
signal_cap = [-2, 2]
chg_func = 'diff'
bullish = True
vol_win = 20
by_asset = False

signal_func = 'ma'
param_rng = [1, 2, 1]
feature = 'us_10y_yield'

freq=''
signal_df = pd.DataFrame(index=df_pxchg.index)

for asset in empiric_assets:
    #feature_ts = df_pxchg[asset].cumsum()
    #feature_ts = df[(asset+'c1', 'close')].dropna() #.pct_change() 
    if by_asset:
        asset_feature = f"{asset.lower()}_{feature}"
    else:
        asset_feature = feature
    #feature_ts = adf["US_IronOre"].dropna()
    #feature_ts = np.log(fut_data[asset].F1) + np.log(fut_data[asset].F3) - 2 * np.log(fut_data[asset].F2)
    feature_ts = np.log(fut_data[asset].F1) - np.log(fut_data[asset].F2)
    feature_ts = feature_ts.cumsum()
    #feature_ts = np.log(fut_data[asset].F1)
    #feature_ts =spot_df[asset_feature].dropna() #.rolling(100).sum()
    #signal_ts = feature_ts.diff(5) #.ewm(3).mean()
    signal_ts = calc_conv_signal(feature_ts, signal_func=signal_func, param_rng=param_rng, signal_cap=signal_cap, vol_win=vol_win) 
    #signal_ts = feature_ts #.pct_change()

    if not bullish:
        signal_ts = -signal_ts    
    
    signal_ts = signal_ts.reindex(index=cdates).ffill().reindex(index=df_pxchg.index)
    #signal_ts = signal_ts.ewm(1).mean()
    #signal_ts = signal_ts.shift(2) 
    signal_df[asset] = signal_ts

signal_df = xs_score(signal_df)
#signal_df = signal_df + xs_demean(signal_df)
#signal_df = signal_buffer(signal_df, 0.3)

#holding = generate_holding_from_signal(signal_df, vol_df, risk_scaling=1.0, asset_scaling=False)
holding = signal_df.div(vol_df).shift(1)
pnl_gross = holding.shift(1).multiply(df_pxchg)
pnl_net = pnl_gross - holding.diff().abs().multiply(2e-4)

met_list = []
for asset in pnl_net:
    sr_ts = calc_perf_by_tenors(pnl_gross[asset], tenors=pnl_tenors, metric='sharpe')
    met_list.append(sr_ts.to_frame(asset))
sr_df = pd.concat(met_list, axis=1)

pnl_per_trade = 100 * 100 * pnl_gross.mean(axis=0) / holding.diff().abs().mean()
turnover = 100 * holding.diff().abs().mean() / holding.abs().mean()


port_pnl = pnl_net.sum(axis=1)
met_list = []
for metric in ['sharpe', 'std']:
    sr_ts = calc_perf_by_tenors(port_pnl, tenors=pnl_tenors, metric=metric)
    sr_ts.index=pnl_tenors
    met_list.append(sr_ts.to_frame(metric))

port_df = pd.concat(met_list, axis=1)

display(port_df.round(3))
display(sr_df.round(3))    
display(pnl_per_trade)
display(turnover)

iplot(port_pnl.cumsum().to_frame("Total"))
iplot(pnl_gross.cumsum())


In [ ]:
lead_lag_config = {
    'll_left': -20,
    'll_right': 120,
    'll_spacing': 5,
    'll_sub_win': [(datetime.date(2018, 1, 1), datetime.date(2022, 12, 31)), 
                   (datetime.date(2023, 1, 1), datetime.date(2025, 12, 31)),],
}

ll_keys = ['fullsample'] + ['%s:%s' % (sd.strftime('%Y-%b-%d'), ed.strftime('%Y-%b-%d')) for sd, ed in lead_lag_config['ll_sub_win']]


ll_left = lead_lag_config['ll_left']
ll_right = lead_lag_config['ll_right']
spacing = lead_lag_config['ll_spacing']

leadlag_df = bt_metrics.lead_lag(ll_limit_left=ll_left, 
                                 ll_limit_right=ll_right,
                                 ll_sub_windows=lead_lag_config['ll_sub_win'])

fig, ax = plt.subplots(len(ll_keys), 1)
fig.set_figheight(15)
fig.set_figwidth(10)

for i, key in enumerate(ll_keys):
    ts = leadlag_df['leadlag_sharpes'].loc[key]
    ts.plot(kind='bar', ax = ax[i], title = f'lead_lag: {key}')
    new_ticks = np.linspace(ll_left, ll_right, (ll_right-ll_left)//spacing+1)
    ax[i].set_xticks(np.interp(new_ticks, ts.index, np.arange(ts.size)))
    ax[i].set_xticklabels(new_ticks)
    ax[i].axvline(x=-ll_left, color='red', linestyle='--')
plt.show()

fig = plt.figure()
ax = fig.add_subplot(111)
ls_pnl = bt_metrics.long_short_pnl()
for key in ls_pnl:
    ax.plot(ls_pnl[key]['portfolio_cumpnl'], '-', label=key)
lines, labels = ax.get_legend_handles_labels()
ax.legend(lines, labels, bbox_to_anchor=(1.04, 1), loc='upper left')
ax.grid()
plt.title("long-short pnl")
plt.show()

lagged = bt_metrics.lagged_pnl(lags=[1, 5, 10, 20, 30, 60, 75, 80])
lagged['cumpnl'].plot()
#print('lagged PNL\n', lagged['sharpe'])
plt.grid()
plt.title('lagged pnl')
plt.show()

smoothed = bt_metrics.smoothed_pnl(smooth_hls=[1, 5, 10, 20, 30, 60, 75, 80])
smoothed['cumpnl'].plot(figsize=(8, 6))
#print('smoothed PNL\n', smoothed['sharpe'])
plt.grid()
plt.title('smoothed pnl')
plt.show()

#tilt_timing = bt_metrics.tilt_timing(tilt_rolling_window=1*244) # default 3 years  tilt_rolling_window = 3 * 244 

seasonal_pnl = bt_metrics.seasonal_pnl()
cumpnl = seasonal_pnl['cumlog_pnl']
cumpnl.set_index(cumpnl.index.astype('str')).plot(rot=30, figsize = (8, 6))
#print('seasonal sharpe stats\n', seasonal_pnl['sharpe_stats'])
plt.grid()
plt.title('monthly pnl')
plt.show()


# monthday_pnl = bt_metrics.monthday_pnl()
# cumpnl = monthday_pnl['cumlog_pnl']
# cumpnl.set_index(cumpnl.index.astype('str')).plot(rot=30, figsize = (8, 6))
# #print('monthday sharpe stats\n', monthday_pnl['sharpe_stats'])
# plt.grid()
# plt.title('monthday pnl')
# plt.show()


week_pnl = bt_metrics.week_pnl()
cumpnl = week_pnl['cumlog_pnl']
cumpnl.set_index(cumpnl.index.astype('str')).plot(rot=30, figsize = (8, 6))
#print('week sharpe stats\n', week_pnl['sharpe_stats'])
plt.grid()
plt.title('weekday pnl')
plt.show()


annual_pnl = bt_metrics.annual_pnl()
cumpnl = annual_pnl['cumlog_pnl']
cumpnl.set_index(cumpnl.index.astype('str')).plot(rot=30, figsize = (8, 6))
#print('annual sharpe stats\n', annual_pnl['sharpe_stats'])
plt.grid()
plt.title('annual pnl')
plt.show()

annual_pnl['cumlog_pnl'].mean(axis=1).plot()
plt.grid()
plt.title('annual averaged profile')
plt.show()

# turnover = bt_metrics.turnover()
# print(turnover)